In [656]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import v2
from torchinfo import summary
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import KFold
from PIL import Image
from scipy import signal
import pandas as pd
import numpy as np
import warnings
import math
import os
warnings.filterwarnings('ignore')

In [657]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 150)

In [658]:
while not os.path.isdir(os.path.join(os.getcwd(), 'data')):
    os.chdir("../") # set cwd to root dir

In [659]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [660]:
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [661]:
writer = SummaryWriter()

In [662]:
class ButterworthFilter(object):
    def __init__(self, cutoff, order, fs):
        self.cutoff = cutoff
        self.order = order
        self.fs = fs

    def __call__(self, input):
        if torch.is_tensor(input):
            input = input.numpy() # convert to numpy array before using signal package

        nyquist = 0.5 * self.fs
        normal_cutoff = self.cutoff / nyquist
        b, a = signal.butter(self.order, normal_cutoff, btype='low', analog=False)
        smoothed_imu_signal = signal.filtfilt(b, a, input)
        
        return torch.from_numpy(smoothed_imu_signal.copy()) # convert back to tensor

In [663]:
class PadTrimToLength(object):
    def __init__(self, padlen):
        self.padlen = padlen

    def __call__(self, input):
        pad = nn.ZeroPad1d((0, max(0, self.padlen - input.shape[1])))(input)
        output = torch.narrow(pad, 1, 0, self.padlen)
        assert output.shape[1] == self.padlen, "Not equal to length of pad"
        return output

In [664]:
transform = v2.Compose([
    ButterworthFilter(cutoff=10.0, order=3, fs=128.0),
    PadTrimToLength(padlen=150),
])

In [665]:
class DUO_GAIT(Dataset):
    def __init__(self, allowed_sensors, participant_id, remove_outliers=True, transform=None, target_transform=None):
        self.participant_id = participant_id
        self.allowed_sensors = allowed_sensors

        lf_control_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_control/sub_{participant_id:02}/left_foot_core_params.csv")
        lf_control_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
        lf_control_df['is_fatigue'] = 0
        lf_control_df['is_right_foot'] = 0
        lf_control_df['Participant'] = participant_id

        self.create_start_end_samples_strides(lf_control_df, is_control=1)

        rf_control_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_control/sub_{participant_id:02}/right_foot_core_params.csv")
        rf_control_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
        rf_control_df['is_fatigue'] = 0
        rf_control_df['is_right_foot'] = 1
        rf_control_df['Participant'] = participant_id

        self.create_start_end_samples_strides(rf_control_df, is_control=1)

        lf_fatigue_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_fatigue/sub_{participant_id:02}/left_foot_core_params.csv")
        lf_fatigue_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
        lf_fatigue_df['is_fatigue'] = 1
        lf_fatigue_df['is_right_foot'] = 0
        lf_fatigue_df['Participant'] = participant_id

        self.create_start_end_samples_strides(lf_fatigue_df, is_control=0)

        rf_fatigue_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_fatigue/sub_{participant_id:02}/right_foot_core_params.csv")
        rf_fatigue_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
        rf_fatigue_df['is_fatigue'] = 1
        rf_fatigue_df['is_right_foot'] = 1
        rf_fatigue_df['Participant'] = participant_id

        self.create_start_end_samples_strides(rf_fatigue_df, is_control=0)

        self.foot_strides_df = pd.concat([lf_control_df, rf_control_df, lf_fatigue_df, rf_fatigue_df], axis=0)

        if remove_outliers:
            self.foot_strides_df = self.foot_strides_df[self.foot_strides_df['is_outlier']==False].reset_index(drop=True)

        self.transform = transform
        self.target_transform = target_transform

    def create_start_end_samples_strides(self, df, is_control):
        df.sort_values(by='stride_index', inplace=True)
        df['start_samples'] = df['ic_samples'].shift(1)
        target_time = df.loc[0, 'start_times']
        
        protocol = "control" if is_control else "fatigue"

        fatigue_df = pd.read_csv(f"data/DUO-GAIT/interim/OG_st_{protocol}/sub_{self.participant_id:02}/LF.csv")
        fatigue_df.rename({ "timestamp": "Time (secs)", "Unnamed: 0": "Sample" }, axis=1, inplace=True)
        fatigue_df['Delta (secs)'] = fatigue_df['Time (secs)'] - fatigue_df['Time (secs)'].min() # delta time

        ts_eq_check = fatigue_df['Delta (secs)'].apply(lambda x: math.isclose(x, target_time, rel_tol=1e-5))
        start_sample = fatigue_df[ts_eq_check]['Sample'].item() - fatigue_df['Sample'].min()

        df.loc[0, 'start_samples'] = start_sample
        df['start_samples'] = df['start_samples'] + fatigue_df['Sample'].min()
        df['end_samples'] = df['ic_samples'] + fatigue_df['Sample'].min() - 1 # make it inclusive for ending samples too

        df['start_samples'] = df['start_samples'].astype(np.int64)
        df['end_samples'] = df['end_samples'].astype(np.int64)

    def __len__(self):
        return len(self.foot_strides_df)

    def __getitem__(self, idx):
        row = self.foot_strides_df.iloc[idx]

        imu_signals_df = pd.DataFrame()

        for sensor_location in self.allowed_sensors:
            protocol = "fatigue" if row['is_fatigue'] else "control"
            sensor_df = pd.read_csv(f"data/DUO-GAIT/interim/OG_st_{protocol}/sub_{self.participant_id:02}/{sensor_location}.csv")
            sensor_df.rename({ "timestamp": "Time (secs)", "Unnamed: 0": "Sample" }, axis=1, inplace=True)
            sensor_df = sensor_df[(sensor_df['Sample'] >= row['start_samples']) & (sensor_df['Sample'] <= row['end_samples'])].reset_index(drop=True)

            for sensor_type in ["Gyr", "Acc"]:
                for direction in ["X", "Y", "Z"]:
                    col_name = f"{sensor_type}{direction}"
                    new_col_name = f"{sensor_location}_{col_name}"

                    imu_signals_df[new_col_name] = sensor_df[col_name]

        imu_signals_np = imu_signals_df.to_numpy().transpose()
        label = row["is_fatigue"]
        
        if self.transform:
            imu_signals_np = self.transform(imu_signals_np)

        if self.target_transform:
            label = self.target_transform(label)

        return imu_signals_np, label

In [666]:
num_participants = 18
ignore_participant_ids = [4,7,16]
allowed_sensors = ["LL"]

In [667]:
k_folds = 5
batch_size = 32
train_percent = 0.90

In [ ]:
for pid in range(1, num_participants+1):
    if pid in ignore_participant_ids: # skip participants that are messy and have missing info
        continue
    
    dataset = DUO_GAIT(allowed_sensors=allowed_sensors, participant_id=pid, transform=transform)
    dataset_size = len(dataset)

    train_size = int(dataset_size * train_percent)
    test_size = dataset_size - train_size
    # 90% for k-fold cv; 10% for final unbiased evaluation

    dataset, test_set = random_split(dataset, [train_size, test_size])

    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42) # 5 fold cv
    for fold, (train_idx, valid_idx) in enumerate(kf.split(dataset)):
        train_set = Subset(dataset, train_idx)
        valid_set = Subset(dataset, valid_idx)

        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
        valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=True)


<class 'torch.utils.data.dataset.Subset'>
[   0    1    2    4    5    6    7    8    9   11   12   13   14   15
   16   17   18   19   20   21   22   24   26   27   28   29   32   33
   34   35   36   37   38   40   41   42   43   45   46   47   48   49
   50   51   52   53   56   57   58   61   62   64   65   68   69   71
   73   74   75   77   78   79   80   81   82   83   84   85   87   89
   90   91   92   93   94   95   97   98   99  102  103  104  105  106
  108  111  112  113  114  115  116  117  118  119  121  122  123  124
  125  126  127  128  129  130  131  132  133  134  135  137  138  140
  141  142  143  144  145  146  147  148  149  150  151  152  153  154
  155  156  157  159  160  161  162  163  164  165  166  167  169  170
  171  172  173  175  176  177  178  179  180  181  182  183  185  186
  187  188  189  190  191  192  193  194  195  196  197  200  201  202
  203  204  205  206  207  211  212  214  216  217  219  220  221  222
  223  224  225  226  227  228  229